# 14 — Dual Paper Trading: Baseline vs ML Challenger

Bu notebook iki ayrı sanal portföyü paralel takip eder:

- **Baseline Robot**
- **ML Challenger:** `Big_Winner_Label_2R + LogisticRegression + Quantile_Q40`

Her iki portföy:

- 500.000 TL ile başlar.
- Aynı final risk ve çıkış kurallarını kullanır.
- Ayrı nakit ve pozisyon state dosyalarına sahiptir.
- Aynı piyasa gerçekleşme fiyatlarıyla kaydedilir.
- Günlük olarak ayrı alış/satış planları üretir.

ML challenger, Baseline Robot'un yerine geçmez. Gerçek yeni out-of-sample
karar yalnızca bu paralel paper-trading kaydıyla verilecektir.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = (
    Path.cwd()
    if (Path.cwd() / "src").exists()
    else Path.cwd().parent
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.config import DataConfig
from src.data_loader import (
    load_bist_tickers,
    download_robot_bundle,
)
from src.data_quality import run_quality_pipeline
from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.ml_dataset import add_meta_features
from src.paper_trading import (
    load_paper_state,
    positions_dataframe,
)
from src.dual_paper import (
    load_challenger_deployment,
    choose_model_ready_signal_date,
    model_score_diagnostic,
    assert_model_scores_available,
    deployment_summary,
    build_challenger_prices,
    create_dual_daily_plan,
    compare_states,
    append_dual_equity_snapshot,
    save_dual_plans,
    record_buy_from_plan,
    record_sell_from_plan,
    persist_portfolio_state,
)


c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Kilitli challenger kararını ve modeli yükle


In [2]:
challenger_deployment, challenger_model = (
    load_challenger_deployment(
        PROJECT_ROOT
    )
)

display(
    deployment_summary(
        challenger_deployment
    )
)


,Target,Model,Filter,Probability_Threshold,Keep_Top_Fraction,Model_Path
0,Big_Winner_Label_2R,LogisticRegression,Quantile_Q40,0.511306,None,c:\Users\okand\Desktop\Projects\Algorithmic Tr...


## 2. Güncel BIST100 ve XU100 verisini indir


In [3]:
DOWNLOAD_START = (
    pd.Timestamp.today().normalize()
    - pd.Timedelta(days=900)
).strftime("%Y-%m-%d")

data_config = DataConfig(
    start=DOWNLOAD_START,
    end=None,
    auto_adjust=True,
    yfinance_repair=False,
)

tickers = load_bist_tickers(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bist100_sirketler.xlsx"
)

raw_stocks, raw_market, download_errors = (
    download_robot_bundle(
        tickers=tickers,
        config=data_config,
    )
)

print("Ticker sayısı:", len(tickers))
print("Hisse satırı:", len(raw_stocks))
print("Endeks satırı:", len(raw_market))
print("İndirme hatası:", len(download_errors))

if not download_errors.empty:
    display(download_errors)


Yahoo Finance verileri: 100%|██████████| 100/100 [00:40<00:00,  2.46it/s]


Ticker sayısı: 100
Hisse satırı: 60382
Endeks satırı: 619
İndirme hatası: 0


## 3. Kalite kontrolü, Robot skorları ve ML özellikleri


In [4]:
stock_quality = run_quality_pipeline(
    raw_stocks,
    config=data_config,
    apply_split_repairs=True,
)

market_quality = run_quality_pipeline(
    raw_market,
    config=data_config,
    apply_split_repairs=True,
)

stock_features = add_indicators(
    stock_quality.clean
)
market_features = add_indicators(
    market_quality.clean
)
market_regime = build_market_regime(
    market_features
)

baseline_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=True,
)

featured_prices = add_meta_features(
    scored_prices=baseline_prices,
    market_features=market_features,
)

latest_stock_date = pd.Timestamp(
    baseline_prices["Date"].max()
)

model_ready_signal_date = (
    choose_model_ready_signal_date(
        featured_prices=featured_prices,
        minimum_coverage_ratio=0.60,
    )
)

prediction_start = (
    model_ready_signal_date
    - pd.Timedelta(days=90)
)

challenger_prices = build_challenger_prices(
    featured_prices=featured_prices,
    fitted_model=challenger_model,
    deployment=challenger_deployment,
    prediction_start=prediction_start,
    prediction_end=model_ready_signal_date,
)

score_diagnostic = model_score_diagnostic(
    baseline_prices=baseline_prices,
    challenger_prices=challenger_prices,
    signal_date=model_ready_signal_date,
    deployment=challenger_deployment,
)

assert_model_scores_available(
    score_diagnostic
)

print("En son hisse fiyat tarihi:", latest_stock_date)
print(
    "Model-ready ortak sinyal tarihi:",
    model_ready_signal_date,
)
print(
    "Veri gecikmesi (takvim günü):",
    (latest_stock_date - model_ready_signal_date).days,
)

print(
    "Model-ready tarihte Baseline AL:",
    baseline_prices.loc[
        baseline_prices["Date"].eq(
            model_ready_signal_date
        ),
        "Signal",
    ].eq("AL").sum(),
)

print(
    "Model-ready tarihte Challenger AL:",
    challenger_prices.loc[
        challenger_prices["Date"].eq(
            model_ready_signal_date
        ),
        "Signal",
    ].eq("AL").sum(),
)

display(score_diagnostic)


En son hisse fiyat tarihi: 2026-07-24 00:00:00
Model-ready ortak sinyal tarihi: 2026-07-23 00:00:00
Veri gecikmesi (takvim günü): 1
Model-ready tarihte Baseline AL: 11
Model-ready tarihte Challenger AL: 4


,Ticker,Score,RSI,ADX,RET_63,RET_126,ML_Probability,ML_Filter_Passed,Challenger_Signal,Locked_Threshold,Distance_To_Threshold,ML_Decision
0,ENJSA.IS,12,56.389826,24.990696,-0.145784,0.165024,0.589969,True,AL,0.511306,0.078663,PASS
1,ODAS.IS,13,66.826671,15.979719,0.301994,0.731061,0.566903,True,AL,0.511306,0.055597,PASS
2,TRENJ.IS,13,65.606691,18.445793,0.077082,0.023599,0.545596,True,AL,0.511306,0.034290,PASS
3,EREGL.IS,14,62.766422,26.433260,0.449870,0.719148,0.528747,True,AL,0.511306,0.017441,PASS
4,PSGYO.IS,13,68.271520,17.609310,0.444088,0.449358,0.508027,False,ALMA,0.511306,-0.003279,REJECT
5,KRDMD.IS,11,56.586644,19.544755,0.180291,0.472067,0.496180,False,ALMA,0.511306,-0.015126,REJECT
6,AKSEN.IS,12,63.361855,37.088056,0.282555,0.515239,0.490465,False,ALMA,0.511306,-0.020841,REJECT
7,TUPRS.IS,12,86.020260,50.013782,0.173507,0.471447,0.477579,False,ALMA,0.511306,-0.033727,REJECT
8,TRMET.IS,12,64.598876,20.958646,-0.015441,0.077232,0.437052,False,ALMA,0.511306,-0.074254,REJECT
9,KUYAS.IS,11,53.208553,24.413995,-0.197368,0.328494,0.349827,False,ALMA,0.511306,-0.161479,REJECT


## 4. İki ayrı paper-trading state dosyasını yükle


In [5]:
DUAL_ROOT = (
    PROJECT_ROOT
    / "results"
    / "paper_trading"
    / "dual"
)

BASELINE_STATE_PATH = (
    DUAL_ROOT
    / "baseline"
    / "state.json"
)
BASELINE_TRADES_PATH = (
    DUAL_ROOT
    / "baseline"
    / "closed_trades.csv"
)

CHALLENGER_STATE_PATH = (
    DUAL_ROOT
    / "challenger"
    / "state.json"
)
CHALLENGER_TRADES_PATH = (
    DUAL_ROOT
    / "challenger"
    / "closed_trades.csv"
)

DUAL_HISTORY_PATH = (
    DUAL_ROOT
    / "dual_equity_history.csv"
)
DUAL_PLANS_DIR = (
    DUAL_ROOT
    / "daily_plans"
)

baseline_state = load_paper_state(
    path=BASELINE_STATE_PATH,
    initial_capital=(
        FINAL_PORTFOLIO_CONFIG.initial_capital
    ),
)

challenger_state = load_paper_state(
    path=CHALLENGER_STATE_PATH,
    initial_capital=(
        FINAL_PORTFOLIO_CONFIG.initial_capital
    ),
)

print("BASELINE POZİSYONLARI")
display(positions_dataframe(baseline_state))

print("CHALLENGER POZİSYONLARI")
display(positions_dataframe(challenger_state))


BASELINE POZİSYONLARI


,Ticker,entry_date,entry_price,shares,stop_loss,highest_price,signal_score


CHALLENGER POZİSYONLARI


,Ticker,entry_date,entry_price,shares,stop_loss,highest_price,signal_score


## 5. İki portföy için eş zamanlı günlük plan üret


In [6]:
dual_plan = create_dual_daily_plan(
    baseline_prices=baseline_prices,
    challenger_prices=challenger_prices,
    baseline_state=baseline_state,
    challenger_state=challenger_state,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    signal_date=model_ready_signal_date,
    minimum_coverage_ratio=0.60,
)

display(dual_plan.summary_comparison)

print("ALIŞ KARŞILAŞTIRMASI")
display(
    dual_plan.buy_comparison.sort_values(
        ["Ticker", "Portfolio"]
    )
)

print("SATIŞ KARŞILAŞTIRMASI")
display(
    dual_plan.sell_comparison.sort_values(
        ["Ticker", "Portfolio"]
    )
)


,Signal_Date,Universe_Rows,Market_Positive,Paper_Cash_TL,Paper_Equity_TL,Open_Positions,Expected_Exits,Available_Slots,Buy_Orders,Sell_Orders,Hold_Positions,Portfolio
0,2026-07-23,98,True,500000.0,500000.0,0,0,6,6,0,0,Baseline_Robot
1,2026-07-23,98,True,500000.0,500000.0,0,0,6,4,0,0,ML_Challenger


ALIŞ KARŞILAŞTIRMASI


,Portfolio,Ticker,Rank,Score,Estimated_Entry,Estimated_Stop,Estimated_Shares,ML_Probability,Portfolio_Count,Appears_In_Both
4,Baseline_Robot,AKSEN.IS,5,12,104.608802,97.383568,479,NaN,1,False
9,ML_Challenger,ENJSA.IS,4,12,108.616802,103.807909,689,0.589969,1,False
0,Baseline_Robot,EREGL.IS,1,14,44.288401,41.842746,1388,NaN,2,True
6,ML_Challenger,EREGL.IS,1,14,44.288401,41.842746,1388,0.528747,2,True
1,Baseline_Robot,ODAS.IS,2,13,9.158280,8.615965,6301,NaN,2,True
7,ML_Challenger,ODAS.IS,2,13,9.158280,8.615965,6301,0.566903,2,True
2,Baseline_Robot,PSGYO.IS,3,13,3.967920,3.654597,11165,NaN,1,False
3,Baseline_Robot,TRENJ.IS,4,13,104.308198,97.904142,535,NaN,2,True
8,ML_Challenger,TRENJ.IS,3,13,104.308198,97.904142,535,0.545596,2,True
5,Baseline_Robot,TUPRS.IS,6,12,315.129000,301.260517,238,NaN,1,False


SATIŞ KARŞILAŞTIRMASI


,Portfolio,Ticker,Action,Reason,Close,Stop_Loss,Trailing_Level


`Appears_In_Both=True` olan hisseler iki portföyün ortak sinyalidir.
Yalnızca Baseline'da bulunanlar ML tarafından elenen sinyallerdir.


In [7]:
plan_paths = save_dual_plans(
    dual_plan=dual_plan,
    output_root=DUAL_PLANS_DIR,
)

for name, path in plan_paths.items():
    print(name, "→", path)


summary → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\paper_trading\dual\daily_plans\2026-07-23\dual_summary.csv
buy_comparison → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\paper_trading\dual\daily_plans\2026-07-23\dual_buy_comparison.csv
sell_comparison → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\paper_trading\dual\daily_plans\2026-07-23\dual_sell_comparison.csv
baseline_buys → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\paper_trading\dual\daily_plans\2026-07-23\baseline_buy_orders.csv
baseline_sells → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\paper_trading\dual\daily_plans\2026-07-23\baseline_sell_orders.csv
challenger_buys → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\paper_trading\dual\daily_plans\2026-07-23\challenger_buy_orders.csv
challenger_sells → c:\Users\okand\Desktop\Projects\Algo

## 6. İki portföyü güncel kapanışlarla değerle


In [8]:
latest_rows = (
    baseline_prices.loc[
        baseline_prices["Date"].eq(
            dual_plan.signal_date
        )
    ]
    .drop_duplicates("Ticker", keep="last")
)

latest_price_map = dict(
    zip(
        latest_rows["Ticker"],
        latest_rows["Close"],
    )
)

dual_equity = compare_states(
    baseline_state=baseline_state,
    challenger_state=challenger_state,
    latest_prices=latest_price_map,
)

display(dual_equity)

append_dual_equity_snapshot(
    comparison=dual_equity,
    signal_date=dual_plan.signal_date,
    path=DUAL_HISTORY_PATH,
)


,Portfolio,Cash,Positions_Value,Equity,Open_Positions,Return_%,Equity_Difference_vs_Baseline_TL
0,Baseline_Robot,500000.0,0.0,500000.0,0,0.0,0.0
1,ML_Challenger,500000.0,0.0,500000.0,0,0.0,0.0


WindowsPath('c:/Users/okand/Desktop/Projects/Algorithmic Trading/BIST-Algo-Trade/results/paper_trading/dual/dual_equity_history.csv')

## 7. Gerçekleşen alışları kaydet

Aşağıdaki örnekler yorum satırındadır. Emir gerçekten dolduktan sonra ilgili
portföyün planını ve state dosyasını kullan.

Aynı hisse iki portföyde de varsa aynı piyasa `FILL_PRICE` değerini kullan;
lot sayıları her portföyün kendi nakit ve risk durumuna göre hesaplanır.


In [9]:
# BASELINE ALIŞ ÖRNEĞİ
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-07-27"
# FILL_PRICE = 70.25
#
# baseline_equity_now = float(
#     dual_equity.loc[
#         dual_equity["Portfolio"].eq("Baseline_Robot"),
#         "Equity",
#     ].iloc[0]
# )
#
# baseline_state, baseline_fill = record_buy_from_plan(
#     state=baseline_state,
#     buy_orders=dual_plan.baseline.buy_orders,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     strategy_config=FINAL_STRATEGY_CONFIG,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
#     current_equity=baseline_equity_now,
# )
#
# persist_portfolio_state(
#     baseline_state,
#     BASELINE_STATE_PATH,
# )
#
# display(pd.DataFrame([baseline_fill]))


In [10]:
# CHALLENGER ALIŞ ÖRNEĞİ
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-07-27"
# FILL_PRICE = 70.25
#
# challenger_equity_now = float(
#     dual_equity.loc[
#         dual_equity["Portfolio"].eq("ML_Challenger"),
#         "Equity",
#     ].iloc[0]
# )
#
# challenger_state, challenger_fill = record_buy_from_plan(
#     state=challenger_state,
#     buy_orders=dual_plan.challenger.buy_orders,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     strategy_config=FINAL_STRATEGY_CONFIG,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
#     current_equity=challenger_equity_now,
# )
#
# persist_portfolio_state(
#     challenger_state,
#     CHALLENGER_STATE_PATH,
# )
#
# display(pd.DataFrame([challenger_fill]))


## 8. Gerçekleşen satışları kaydet


In [11]:
# BASELINE SATIŞ ÖRNEĞİ
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-08-10"
# FILL_PRICE = 76.80
#
# baseline_state, baseline_trade = record_sell_from_plan(
#     state=baseline_state,
#     sell_orders=dual_plan.baseline.sell_orders,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
#     trades_path=BASELINE_TRADES_PATH,
# )
#
# persist_portfolio_state(
#     baseline_state,
#     BASELINE_STATE_PATH,
# )
#
# display(pd.DataFrame([baseline_trade]))


In [12]:
# CHALLENGER SATIŞ ÖRNEĞİ
#
# TICKER = "AKBNK.IS"
# FILL_DATE = "2026-08-10"
# FILL_PRICE = 76.80
#
# challenger_state, challenger_trade = record_sell_from_plan(
#     state=challenger_state,
#     sell_orders=dual_plan.challenger.sell_orders,
#     ticker=TICKER,
#     fill_date=FILL_DATE,
#     fill_price=FILL_PRICE,
#     portfolio_config=FINAL_PORTFOLIO_CONFIG,
#     trades_path=CHALLENGER_TRADES_PATH,
# )
#
# persist_portfolio_state(
#     challenger_state,
#     CHALLENGER_STATE_PATH,
# )
#
# display(pd.DataFrame([challenger_trade]))


## Günlük çalışma düzeni

Piyasa kapandıktan sonra:

1. Bölüm 1–6'yı çalıştır.
2. Baseline ve Challenger alış/satış farklarını incele.
3. Ertesi gün gerçekleşen emirleri Bölüm 7–8 ile kaydet.
4. İki portföyde aynı emir varsa aynı gerçekleşme fiyatını kullan.
5. ML modelini, threshold'u veya strateji parametrelerini paper-trading
   değerlendirmesi tamamlanmadan değiştirme.

İlk değerlendirme için asgari hedef:

- En az 50 kapalı challenger işlemi
- Tercihen 3–6 aylık piyasa dönemi
- Yükseliş, yatay ve düzeltme günlerinin birlikte görülmesi
